# Conditional diffusion: wave fields → surface-current ensemble

This notebook uses the existing `WaveCurrentDataset` and a compact 2-D EDM model. Dataset time/calendar metadata is not used; `sigma` below is diffusion noise level, not physical time.

In [1]:
from pathlib import Path
import csv
import sys
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

project = Path.cwd()
if project.name == "notebooks":
    project = project.parent
sys.path.insert(0, str(project))

from ocean_velocity.config import load_config
from ocean_velocity.train import make_dataset
from ocean_velocity import ConditionalDiffusionUNet, EDMPreconditioner, edm_sample
from ocean_velocity.diffusion import edm_loss
from ocean_velocity.diffusion_train import train_diffusion_epoch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Reuse the existing data configuration

Change the config path if needed. Normalized wave fields become the condition and normalized currents become the generated target.

In [2]:
config = load_config(project / "configs" / "agulhas-5x5.json")
train_dataset = make_dataset(config, "train")
validation_dataset = make_dataset(config, "validation")
# Validate on held-out dates using the same spatial patch size as training.
validation_dataset.patch_size = train_dataset.patch_size
train_loader = DataLoader(
    train_dataset, batch_size=148, shuffle=True, num_workers=8,
    pin_memory=True, persistent_workers=True, prefetch_factor=2,
)
validation_loader = DataLoader(
    validation_dataset, batch_size=48, shuffle=False, num_workers=8,
    pin_memory=True, persistent_workers=True, prefetch_factor=2,
)
batch = next(iter(train_loader))
{name: tuple(value.shape) for name, value in batch.items() if hasattr(value, "shape")}

/ext3/miniforge3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/ext3/miniforge3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


{'x': (148, 3, 60, 60),
 'y': (148, 2, 60, 60),
 'metadata': (148, 2),
 'metadata_valid': (148,),
 'valid_mask': (148, 1, 60, 60),
 'index': (148,)}

## Build a lightweight denoiser

`base_channels=16` is convenient for notebook experiments. Try 32 for a fuller run. Because configured targets are standardized, `sigma_data=1.0` is the natural starting value.

In [3]:
backbone = ConditionalDiffusionUNet(
    target_channels=len(config["data"]["target_variables"]),
    condition_channels=len(config["data"]["input_variables"]),
    base_channels=16,
    channel_multipliers=(1, 2, 4),
)
model = EDMPreconditioner(backbone, sigma_data=1.0).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-5)
sum(parameter.numel() for parameter in model.parameters())

362994

## Train

Training currently uses a complete 1,460-batch pass. Validation is evaluated on a fixed set of held-out 48×48 patches and diffusion noise draws, making losses comparable from epoch to epoch. Increase `validation_batches` for a more precise (but slower) estimate.

In [ ]:
@torch.no_grad()
def validation_diffusion_loss(net, loader, device, max_batches=16):
    net.eval()
    losses = []
    # Fix validation crops and diffusion noise without changing training RNG state.
    devices = [device.index or 0] if device.type == "cuda" else []
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(0)
        for validation_batch in loader:
            condition = validation_batch["x"].to(device)
            target = validation_batch["y"].to(device)
            mask = validation_batch["valid_mask"].to(device)
            losses.append(float(edm_loss(net, target, condition, mask)))
            if len(losses) >= max_batches:
                break
    return sum(losses) / len(losses)


validation_batches = 16
loss_history = []
history_path = project / "outputs" / "diffusion_loss_history.csv"
history_path.parent.mkdir(parents=True, exist_ok=True)

with history_path.open("w", newline="") as history_file:
    history_writer = csv.DictWriter(
        history_file, fieldnames=["epoch", "train_loss", "validation_loss"]
    )
    history_writer.writeheader()

    for epoch in range(2000):
        train_loss = train_diffusion_epoch(
            model, train_loader, optimizer, device, max_batches=1460
        )
        validation_loss = validation_diffusion_loss(
            model, validation_loader, device, max_batches=validation_batches
        )
        record = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "validation_loss": validation_loss,
        }
        loss_history.append(record)
        history_writer.writerow(record)
        history_file.flush()  # Preserve every completed epoch if training is interrupted.
        print(
            f"epoch={epoch + 1} train_loss={train_loss:.5f} "
            f"validation_loss={validation_loss:.5f}"
        )

print(f"Saved loss history to {history_path}")

# torch.save({"model": model.state_dict(), "config": config}, "diffusion.pt")

/ext3/miniforge3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


epoch=1 train_loss=0.95752 validation_loss=0.90391


## Plot saved training history

This cell reloads the CSV, so it also works after restarting the kernel or interrupting training. The figure is displayed and saved beside the CSV.

In [ ]:
with history_path.open(newline="") as history_file:
    saved_history = list(csv.DictReader(history_file))

epochs = [int(row["epoch"]) for row in saved_history]
train_losses = [float(row["train_loss"]) for row in saved_history]
validation_losses = [float(row["validation_loss"]) for row in saved_history]

fig, axis = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
axis.plot(epochs, train_losses, label="Train")
axis.plot(epochs, validation_losses, label="Validation")
axis.set(xlabel="Epoch", ylabel="EDM loss", title="Diffusion training history")
axis.grid(alpha=0.25)
axis.legend()
loss_plot_path = history_path.with_suffix(".png")
fig.savefig(loss_plot_path, dpi=160)
plt.show()
print(f"Saved loss plot to {loss_plot_path}")

## Draw conditional samples

Repeat conditions to draw an ensemble for the same wave state. Samples are returned in normalized target space; multiply by target standard deviation and add its mean for physical units.

In [ ]:
model.eval()
plot_batch = next(iter(validation_loader))
condition_single = plot_batch["x"][:1].to(device)
target_single = plot_batch["y"][:1].to(device)
ensemble_size = 4
condition = condition_single.repeat(ensemble_size, 1, 1, 1)
with torch.no_grad():
    samples = edm_sample(model, condition, num_steps=18)

stats = config["data"]["normalization"]["target"]
mean = torch.tensor(stats["mean"], device=device)[None, :, None, None]
std = torch.tensor(stats["std"], device=device)[None, :, None, None]
samples_physical = samples * std + mean
target_physical = target_single * std + mean

input_stats = config["data"]["normalization"]["input"]
input_mean = torch.tensor(input_stats["mean"], device=device)[None, :, None, None]
input_std = torch.tensor(input_stats["std"], device=device)[None, :, None, None]
condition_physical = condition_single * input_std + input_mean
samples_physical.shape

## Plot the condition, truth, and generated ensemble

The first figure shows the held-out wave-condition patch in physical units. The second compares the true current, ensemble mean, and each stochastic diffusion sample. Each current component uses a common color scale across columns.

In [ ]:
wave_names = config["data"]["input_variables"]
current_names = config["data"]["target_variables"]
waves = condition_physical[0].detach().cpu()
truth = target_physical[0].detach().cpu()
ensemble = samples_physical.detach().cpu()

fig, axes = plt.subplots(1, len(wave_names), figsize=(13, 3.6), constrained_layout=True)
for channel, (axis, name) in enumerate(zip(axes, wave_names)):
    image = axis.imshow(waves[channel], origin="lower", cmap="viridis")
    axis.set_title(f"Condition: {name}")
    axis.set_xticks([])
    axis.set_yticks([])
    fig.colorbar(image, ax=axis, shrink=0.8)
plt.show()

column_titles = ["Truth", "Ensemble mean"] + [
    f"Sample {index + 1}" for index in range(ensemble_size)
]
fig, axes = plt.subplots(
    len(current_names), len(column_titles),
    figsize=(3 * len(column_titles), 3 * len(current_names)),
    constrained_layout=True,
    squeeze=False,
)
for channel, name in enumerate(current_names):
    panels = [truth[channel], ensemble[:, channel].mean(0)] + [
        ensemble[index, channel] for index in range(ensemble_size)
    ]
    scale = torch.stack(panels)
    vmin, vmax = torch.quantile(scale, torch.tensor([0.01, 0.99])).tolist()
    for column, (axis, panel, title) in enumerate(zip(axes[channel], panels, column_titles)):
        image = axis.imshow(panel, origin="lower", cmap="RdBu_r", vmin=vmin, vmax=vmax)
        if channel == 0:
            axis.set_title(title)
        if column == 0:
            axis.set_ylabel(name)
        axis.set_xticks([])
        axis.set_yticks([])
    fig.colorbar(image, ax=axes[channel].tolist(), shrink=0.75, label=name)
plt.show()